# Loading the data

In [32]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
import numpy as np

In [3]:
df_train = pd.read_csv('https://raw.githubusercontent.com/data-bootcamp-v4/data/main/sales.csv')
df_predict = pd.read_csv('https://raw.githubusercontent.com/data-bootcamp-v4/data/main/ironkaggle_notarget.csv')

# Data Wrangling 

In [10]:
print("TRAIN data")
print(df_train.shape)
print(df_train.dtypes)
print(df_train.head())

TRAIN data
(640840, 10)
True_index              int64
Store_ID                int64
Day_of_week             int64
Date                   object
Nb_customers_on_day     int64
Open                    int64
Promotion               int64
State_holiday          object
School_holiday          int64
Sales                   int64
dtype: object
   True_index  Store_ID  Day_of_week        Date  Nb_customers_on_day  Open  \
0           0       625            3  2013-11-06                  641     1   
1           1       293            2  2013-07-16                  877     1   
2           2        39            4  2014-01-23                  561     1   
3           3       676            4  2013-09-26                 1584     1   
4           4       709            3  2014-01-22                 1477     1   

   Promotion State_holiday  School_holiday  Sales  
0          1             0               0   7293  
1          1             0               1   7060  
2          1             0     

In [13]:
df_train.isnull().sum()

True_index             0
Store_ID               0
Day_of_week            0
Date                   0
Nb_customers_on_day    0
Open                   0
Promotion              0
State_holiday          0
School_holiday         0
Sales                  0
dtype: int64

In [11]:
print("PREDICTION DATA")
print(df_predict.shape)
print(df_predict.dtypes)
print(df_predict.head())

PREDICTION DATA
(71205, 9)
True_index              int64
Store_ID                int64
Day_of_week             int64
Date                   object
Nb_customers_on_day     int64
Open                    int64
Promotion               int64
State_holiday          object
School_holiday          int64
dtype: object
   True_index  Store_ID  Day_of_week        Date  Nb_customers_on_day  Open  \
0           7       764            4  2013-12-26                    0     0   
1          19        22            3  2013-05-22                  449     1   
2          31      1087            6  2013-06-29                  622     1   
3          45       139            6  2013-08-17                  314     1   
4          56       568            1  2014-04-07                  356     1   

   Promotion State_holiday  School_holiday  
0          0             c               1  
1          0             0               1  
2          0             0               0  
3          0             0        

In [14]:
df_predict.isnull().sum()

True_index             0
Store_ID               0
Day_of_week            0
Date                   0
Nb_customers_on_day    0
Open                   0
Promotion              0
State_holiday          0
School_holiday         0
dtype: int64

In [15]:
# ── Step 3: Check value counts for categorical/binary columns ─────────────────
print("=== State_holiday (df_train) ===")
print(df_train["State_holiday"].value_counts())

print("\n=== State_holiday (df_predict) ===")
print(df_predict["State_holiday"].value_counts())

print("\n=== Open (df_train) ===")
print(df_train["Open"].value_counts())

print("\n=== Open (df_predict) ===")
print(df_predict["Open"].value_counts())

print("\n=== Promotion (df_train) ===")
print(df_train["Promotion"].value_counts())

=== State_holiday (df_train) ===
State_holiday
0    621160
a     12842
b      4214
c      2624
Name: count, dtype: int64

=== State_holiday (df_predict) ===
State_holiday
0    69050
a     1405
b      475
c      275
Name: count, dtype: int64

=== Open (df_train) ===
Open
1    532016
0    108824
Name: count, dtype: int64

=== Open (df_predict) ===
Open
1    59105
0    12100
Name: count, dtype: int64

=== Promotion (df_train) ===
Promotion
0    396220
1    244620
Name: count, dtype: int64


In [16]:
# ── Step 4: Check target variable distribution ────────────────────────────────
print("=== Sales distribution (df_train) ===")
print(df_train["Sales"].describe())

print("\n=== Any Sales = 0 when Open = 1? (should be 0) ===")
print(df_train[(df_train["Open"] == 1) & (df_train["Sales"] == 0)].shape[0])

print("\n=== Any Sales > 0 when Open = 0? (should be 0) ===")
print(df_train[(df_train["Open"] == 0) & (df_train["Sales"] > 0)].shape[0])

=== Sales distribution (df_train) ===
count    640840.000000
mean       5777.469011
std        3851.338083
min           0.000000
25%        3731.000000
50%        5746.000000
75%        7860.000000
max       41551.000000
Name: Sales, dtype: float64

=== Any Sales = 0 when Open = 1? (should be 0) ===
31

=== Any Sales > 0 when Open = 0? (should be 0) ===
0


In [17]:
# ── Step 5: Check Date format ─────────────────────────────────────────────────
print("=== Date sample (df_train) ===")
print(df_train["Date"].head(10))
print("Min:", df_train["Date"].min(), "| Max:", df_train["Date"].max())

print("\n=== Date sample (df_predict) ===")
print(df_predict["Date"].head(10))

=== Date sample (df_train) ===
0    2013-11-06
1    2013-07-16
2    2014-01-23
3    2013-09-26
4    2014-01-22
5    2014-10-04
6    2013-06-05
7    2013-02-06
8    2013-10-21
9    2014-06-26
Name: Date, dtype: object
Min: 2013-01-01 | Max: 2015-07-31

=== Date sample (df_predict) ===
0    2013-12-26
1    2013-05-22
2    2013-06-29
3    2013-08-17
4    2014-04-07
5    2015-06-20
6    2014-12-31
7    2014-06-23
8    2013-10-17
9    2013-11-23
Name: Date, dtype: object


In [18]:
# ── Step 6: Drop anomalies (Open=1 but Sales=0) ───────────────────────────────
print(f"Rows before: {df_train.shape[0]}")
df_train = df_train[~((df_train["Open"] == 1) & (df_train["Sales"] == 0))]
print(f"Rows after dropping Open=1 & Sales=0: {df_train.shape[0]}")

Rows before: 640840
Rows after dropping Open=1 & Sales=0: 640809


In [19]:
# ── Step 7: Parse Date into features (apply to both datasets) ─────────────────
for df in [df_train, df_predict]:
    df["Date"] = pd.to_datetime(df["Date"])
    df["Month"] = df["Date"].dt.month
    df["Year"] = df["Date"].dt.year
    df["Day"] = df["Date"].dt.day

df_train = df_train.drop(columns=["Date"])
df_predict = df_predict.drop(columns=["Date"])

print("Date parsed. New columns:", [c for c in df_train.columns if c in ["Month","Year","Day"]])

Date parsed. New columns: ['Month', 'Year', 'Day']


In [20]:
# ── Step 8: Encode State_holiday ──────────────────────────────────────────────
mapping = {"0": 0, "a": 1, "b": 2, "c": 3}

df_train["State_holiday"] = df_train["State_holiday"].astype(str).map(mapping)
df_predict["State_holiday"] = df_predict["State_holiday"].astype(str).map(mapping)

print("=== State_holiday after encoding (df_train) ===")
print(df_train["State_holiday"].value_counts())

=== State_holiday after encoding (df_train) ===
State_holiday
0    621129
1     12842
2      4214
3      2624
Name: count, dtype: int64


In [21]:
# ── Step 9: Final check before modeling ───────────────────────────────────────
print("=== df_train shape ===", df_train.shape)
print("=== df_predict shape ===", df_predict.shape)
print("\n=== Any nulls left in df_train? ===")
print(df_train.isnull().sum())
print("\n=== df_train dtypes ===")
print(df_train.dtypes)

=== df_train shape === (640809, 12)
=== df_predict shape === (71205, 11)

=== Any nulls left in df_train? ===
True_index             0
Store_ID               0
Day_of_week            0
Nb_customers_on_day    0
Open                   0
Promotion              0
State_holiday          0
School_holiday         0
Sales                  0
Month                  0
Year                   0
Day                    0
dtype: int64

=== df_train dtypes ===
True_index             int64
Store_ID               int64
Day_of_week            int64
Nb_customers_on_day    int64
Open                   int64
Promotion              int64
State_holiday          int64
School_holiday         int64
Sales                  int64
Month                  int32
Year                   int32
Day                    int32
dtype: object


In [22]:
# ── Step 10: Filter out Open=0 rows for training ──────────────────────────────
df_train_open = df_train[df_train["Open"] == 1].copy()
print(f"Training rows (open only): {df_train_open.shape[0]}")

# ── Step 11: Define features and target ───────────────────────────────────────
# Drop True_index (just an ID), Open (always 1 now), and Sales (target)
FEATURES = ["Store_ID", "Day_of_week", "Nb_customers_on_day", 
            "Promotion", "State_holiday", "School_holiday", 
            "Month", "Year", "Day"]

TARGET = "Sales"

X = df_train_open[FEATURES]
y = df_train_open[TARGET]

print(f"\nFeatures: {FEATURES}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\ny (Sales) stats:\n{y.describe()}")

Training rows (open only): 531985

Features: ['Store_ID', 'Day_of_week', 'Nb_customers_on_day', 'Promotion', 'State_holiday', 'School_holiday', 'Month', 'Year', 'Day']
X shape: (531985, 9)
y shape: (531985,)

y (Sales) stats:
count    531985.000000
mean       6959.657210
std        3104.877712
min          46.000000
25%        4861.000000
50%        6373.000000
75%        8365.000000
max       41551.000000
Name: Sales, dtype: float64


# let's test Random Forest 

In [23]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [25]:
X_train.shape

(425588, 9)

In [26]:
X_test.shape

(106397, 9)

In [30]:
# ── Step 13: Train model ──────────────────────────────────────────────────────
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [34]:
# ── Step 14: Evaluate ─────────────────────────────────────────────────────────
y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_pred, y_test))
print("RMSE:", root_mean_squared_error(y_pred, y_test))
print("R2 score:", model.score(X_test, y_test))

MAE: 645.8071805596023
RMSE: 955.2413855470769
R2 score: 0.9048633520413166


In [35]:
# ── Step 15: Prepare df_predict features ──────────────────────────────────────
predict_features = df_predict[FEATURES]

print("df_predict features shape:", predict_features.shape)
print("Any nulls?", predict_features.isnull().sum().sum())

df_predict features shape: (71205, 9)
Any nulls? 0


In [36]:
# ── Step 16: Predict ──────────────────────────────────────────────────────────
y_pred_final = model.predict(predict_features)

# Force 0 for closed shops
y_pred_final[df_predict["Open"] == 0] = 0

print("Predictions shape:", y_pred_final.shape)
print("Sample predictions:", y_pred_final[:5])

Predictions shape: (71205,)
Sample predictions: [   0.   3593.65 6311.62 3746.01 3797.09]


In [37]:
# ── Step 17: Build submission CSV ─────────────────────────────────────────────
submission = pd.DataFrame({
    "True_index": df_predict["True_index"],
    "Sales": y_pred_final
})

print(submission.head(10))
print("Shape:", submission.shape)

   True_index    Sales
0           7     0.00
1          19  3593.65
2          31  6311.62
3          45  3746.01
4          56  3797.09
5          57  5672.14
6          61  5838.99
7          63  7279.79
8          79  5319.56
9          81  9690.49
Shape: (71205, 2)


In [39]:
# ── Step 18: Export ───────────────────────────────────────────────────────────
submission.to_csv("my_predictions.csv", index=False)
print("my_predictions.csv saved!")

my_predictions.csv saved!


# Hyperparameter tuning

In [40]:
from sklearn.model_selection import GridSearchCV

In [41]:
param_grid = {
    "n_estimators": [100, 200, 500],
    "max_depth": [None, 20, 30],
    "min_samples_leaf": [1, 2]
}

In [42]:
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=3,
    n_jobs=-1
)

In [ ]:
grid_search.fit(X_train, y_train)

My laptop crashed, so to be continued ...